In [ ]:
from pathlib import Path
from typing import Literal

import geopandas as gpd
import pandas as pd
import pendulum
import plotly.express as px

from un_llm_analysis.database import get_db_connection
from un_llm_analysis.config import APP_DIR

In [ ]:
conn = get_db_connection()
query = """select briefing_id, section_id, country, iso_2, iso_3, model from country;"""
with conn:
    cursor = conn.cursor()
    cursor.execute(query)
    rows = cursor.fetchall()
    column_names = [x[0] for x in cursor.description]
    cursor.close()

df_country = pd.DataFrame(rows, columns=column_names)

df_country["date"] = df_country["briefing_id"].apply(
    lambda x: pendulum.from_format(str(x), "YYYYMMDD").format("YYYY-MM-DD")
)
df_country["week"] = df_country["briefing_id"].apply(
    lambda x: pendulum.from_format(str(x), "YYYYMMDD").start_of("week").format("YYYY-MM-DD")
)
df_country["month"] = df_country["briefing_id"].apply(
    lambda x: pendulum.from_format(str(x), "YYYYMMDD").start_of("month").format("YYYY-MM-DD")
)
df_country["year"] = df_country["briefing_id"].apply(
    lambda x: pendulum.from_format(str(x), "YYYYMMDD").start_of("year").format("YYYY-MM-DD")
)

In [ ]:
def create_heatmap(level: Literal["day", "week", "month", "year"] = "month"):
    x = (
        df_country[[level, "country"]]
        .value_counts()
        .reset_index()
        .pivot(index=level, columns=["country"])
        .fillna(0)
        .reset_index()[level]
        .to_list()
    )

    y = sorted(df_country["country"].unique())
    z = (
        df_country[[level, "country"]]
        .value_counts()
        .reset_index()
        .pivot(index=level, columns=["country"])
        .fillna(0)
        .reset_index()
        .T.values[1:]
    )
    fig = px.imshow(
        z,
        text_auto=True,
        aspect="auto",
        labels=dict(x="Date", y="Country", color="Mentions"),
        x=x,
        y=y,
        color_continuous_scale="RdBu_r",
    )
    # fig.update_layout(yaxis={"dtick": 1})
    fig.update_layout(
        {
            "yaxis": {
                "range": [0, len(y)],
                "tickmode": "array",
                "tickvals": [*range(len(y))],
                "ticktext": y,
            },
            "height": 1700,
            # "width": 200 + (len(x) * 50),
            # "width": 200 + (len(x) * 50),
            "template": "plotly_dark",
        }
    )
    return fig


fig = create_heatmap("month")
fig.show()

In [ ]:
geo_data = gpd.read_file(Path(APP_DIR / "static" / "ne_110m_admin_0_countries.zip"))
gdf_map = geo_data[["ADM0_A3", "geometry"]].merge(
    df_country[["iso_3"]].value_counts().reset_index(),
    how="right",
    left_on="ADM0_A3",
    right_on="iso_3",
)
gdf_map.plot(column="count", legend=True, cmap="OrRd", figsize=(15, 20))

In [ ]:
# https://stackoverflow.com/questions/63245563/make-this-notebook-trusted-to-load-map-file-trust-notebook
gdf_map.explore(column="count")